# Estimate Truck-Type Percentages from CSF2TDM

This notebook calculates IX, XI, and XX truck trips from CSF2TDM, 2020 OD Trips by truck class (small, medium, and large) and uses these totals to calculate truck-type shares relative to the total CTPP external trip volume. The resulting shares can be used to update the CTPP truck trip distribution so it is consistent with CSF2TDM truck demand estimates.

Note: Because very small trucks are not represented within CSF2TDM, no revised share can be estimated. The existing percentage is therefore maintained.

These shares are used in model-files/scripts/nonres/IxTimeOfDay.job

In [ ]:
import pandas as pd 
import openmatrix as omx
import numpy as np

from collections import defaultdict

In [ ]:
# CTPP 2012-2016 data
path = "C:/temp/mtc_cube_runs/TM-1.6/nonres/ixDailyx4.omx"  # converted from 2023_TM161_IPA_35/nonres/ixDailyx4.tpp
ctpp = omx.open_file(path, 'r')

# CSF2TDM OD Trips - TAZ1475 Format
path = "../data/interim/matrix_projection/projected_matrices/SwTazTln_to_TMTaz_TRIPS_FFM_2020.omx"
csf2tdm = omx.open_file(path, 'r')

outpath = "../data/interim/notebook_outputs/truck_types_shares.csv"

# CTPP IX-XI-XX Trips

In [ ]:
n_taz = 1454

In [ ]:
m = np.array(ctpp['ix_daily_total'])

# Internal-External Total trips
ctpp_daily_ix = m[n_taz:][:,:n_taz].sum()
ctpp_daily_xi = m[:n_taz][:,n_taz:].sum()
ctpp_daily_xx = m[n_taz:][:,n_taz:].sum()
ctpp_total = ctpp_daily_ix + ctpp_daily_xi + ctpp_daily_xx

print(f"Daily IX: {ctpp_daily_ix:,.0f}")
print(f"Daily XI: {ctpp_daily_xi:,.0f}")
print(f"Daily XX: {ctpp_daily_xx:,.0f}")
print(f"Total Daily XI-IX-XX: {ctpp_total:,.0f}")

In [ ]:
# From TM-1.6 percentages in: https://github.com/BayAreaMetro/travel-model-one/blob/2b34114301d1e4bfedb65e81203e2cc568da630b/model-files/scripts/nonres/IxTimeOfDay.job#L44-L47
IX_EX_TRK_VSM_SHARE = 0.162  # move this share to very small trucks
IX_EX_TRK_SML_SHARE = 0.028  # move this share to small trucks
IX_EX_TRK_MED_SHARE = 0.004  # move this share to medium trucks
IX_EX_TRK_LRG_SHARE = 0.006  # move this share to large trucks

# Truck Trips TM-1.6
print(f"Very Small XI-IX-XX: {IX_EX_TRK_VSM_SHARE * ctpp_total:,.0f}")
print(f"Small XI-IX-XX: {IX_EX_TRK_SML_SHARE * ctpp_total:,.0f}")
print(f"Medium XI-IX-XX: {IX_EX_TRK_MED_SHARE * ctpp_total:,.0f}")
print(f"Large XI-IX-XX: {IX_EX_TRK_LRG_SHARE * ctpp_total:,.0f}")

# Update IX-XI-XX Percentages

In [ ]:
truck_count = defaultdict(int)
truck_keys = {
    "HT": "Large", 
    "MT": "Medium", 
    "LT": "Small",
}

for matrix_name in csf2tdm.list_matrices():
    truck_type = truck_keys[matrix_name[:2]]
    m = np.array(csf2tdm[matrix_name])
    daily_ix = m[n_taz:][:,:n_taz].sum()
    daily_xi = m[:n_taz][:,n_taz:].sum()
    daily_xx = m[n_taz:][:,n_taz:].sum()
    total = daily_ix + daily_xi + daily_xx
    truck_count[truck_type] += total

# # Print summary (your existing output)
# for key, value in truck_count.items():
#     new_pct = value / ctpp_total
#     print(f"{key}: {value:,.0f} ix-xi-xx truck trips ({new_pct:.1%} of CTPP total)")

truck_count_df = pd.DataFrame(
    [
        {
            "TruckType": key,
            "Trips": value,
            "CTPPTotal": ctpp_total,
            "PctOfCTPP": value / ctpp_total
        }
        for key, value in truck_count.items()
    ]
)

# Still print your summary
for _, row in truck_count_df.iterrows():
    print(
        f"{row['TruckType']}: "
        f"{row['Trips']:,.0f} ix-xi-xx truck trips "
        f"({row['PctOfCTPP']:.1%} of CTPP total)"
    )

truck_count_df.to_csv(outpath, index=False)